In [1]:
 #pip install langchain langchain-openai langchain-community langgraph python-dotenv faiss-cpu pypdf

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [5]:
load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
# to load intro-to-ml document
loader = PyPDFLoader("intro-to-ml.pdf") #tells langchain to read this document
docs = loader.load()#1)reads this document 2)splits it into pages 3) returns list of doc objs[page1,page2....]

In [ ]:
len(docs) # to check lent of the that list

234

In [10]:
#defining splitter so that chunk will be of 1000 ch and overlap of 200 ch
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
#using defined splitter to split this docs list further into chunks
chunks = splitter.split_documents(docs)

In [ ]:
#selecting embedding model 
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
#embedding chunks using selected embedding model and storing in vector db FAISS 
vector_store = FAISS.from_documents(chunks, embeddings)


In [ ]:
vector_store 

In [ ]:
#defining retriever by search type and how many chunks(paragraphs) to retrieve
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
@tool
def rag_tool(query):

    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.
    """
    #retrieves top 4 related chunks
    result = retriever.invoke(query)
    
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):

    messages = state['messages']

    response = llm_with_tools.invoke(messages)

    return {'messages': [response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()
chatbot

In [ ]:
result = chatbot.invoke({"messages": [HumanMessage(content=("Using the pdf notes, explain how to find the ideal value of K in KNN"))]})

In [ ]:
print(result['messages'][-1].content)